## Parte 1: Explicar e explorar

Para **cada** bloco de código desta aula:

1. **Explicar**: escreva, com suas palavras, o que cada linha faz e por que ela existe.
2. **Explorar**: rode o código, **altere** algo (um valor, um argumento, um termo) e registre *o que mudou no resultado*.
3. **Prever**: antes de rodar a versão alterada, escreva o que você *espera* que aconteça, e compare.


## Bloco 1: O corpus e o bit

In [ ]:
docs <- c(
  d1 = "recuperacao de informacao ordena documentos por relevancia",
  d2 = "o modelo de espaco vetorial representa documentos como vetores",
  d3 = "bm25 e um modelo probabilistico de ranqueamento de texto",
  d4 = "aprendizado estatistico fundamenta a recuperacao moderna",
  d5 = "o indice invertido acelera a busca em muitos documentos",
  d6 = "embeddings capturam a semantica de palavras e documentos",
  d7 = "a avaliacao mede a relevancia dos resultados da busca",
  d8 = "ciencia de dados combina estatistica e programacao"
)
length(docs)

N <- length(docs)
log2(N)   # perguntas necessárias para achar 1 entre N

[1] 8
[1] 3


### 1. O que cada linha faz:
- `docs <- c(...)`: recria o mesmo corpus de brinquedo de 8 documentos usado desde a Aula 01, cada um indexado por seu nome (`d1` a `d8`).
- `length(docs)`: conta quantos documentos existem no corpus; retorna 8.
- `N <- length(docs)`: guarda esse total numa variável, porque vamos usá-lo repetidas vezes como "tamanho do conjunto de mensagens possíveis" (no sentido de Shannon).
- `log2(N)`: calcula quantas vezes dá para dividir N por 2, ou seja, quantas perguntas de sim/não, no pior caso, são necessárias para achar **um** documento específico entre os N. Com N=8, o resultado é 3 (8 → 4 → 2 → 1).

### 2. Prever e Explorar
- E se o corpus tivesse o dobro de documentos (16, duplicando os 8 atuais)?

**Previsão:** duplicar N deveria aumentar `log2(N)` em exatamente 1 bit (de 3 para 4), porque log2 de um valor que dobra sempre soma 1.

In [ ]:
docs16 <- c(docs, setNames(docs, paste0(names(docs), "b")))
length(docs16)
log2(length(docs16))
log2(length(docs))  # para comparar

[1] 16
[1] 4
[1] 3


**Resultado:** exatamente como previsto: `length(docs16)` foi para 16 e `log2(16) = 4`, um bit a mais que os 3 bits de antes. Isso confirma visualmente por que dobrar o corpus custa sempre +1 bit, não o dobro: log2 comprime crescimento multiplicativo em soma.

## Bloco 2: Quanto vale uma pista

In [ ]:
tok <- function(x) unlist(strsplit(x, " "))
contem <- function(t) names(docs)[sapply(docs, function(d) t %in% tok(d))]
contem("busca")

log2(8 / 2)  # de 8 candidatos, a pista deixou 2

[1] "d5" "d7"
[1] 2


### 1. O que cada linha faz:
- `tok <- function(x) unlist(strsplit(x, " "))`: uma tokenização simples por espaço (sem `tolower`, já que o corpus já está em minúsculas).
- `contem <- function(t) ...`: para um termo `t`, percorre cada documento com `sapply`, testa se `t` está entre os tokens dele (`t %in% tok(d)`), e devolve os **nomes** dos documentos onde isso é verdade; funciona como uma busca booleana simplificada.
- `contem("busca")`: retorna `d5` e `d7`, os dois únicos documentos que citam "busca".
- `log2(8 / 2)`: como a pista reduziu o conjunto de candidatos de 8 para 2, ela "pagou" `log2(8/2) = 2` bits dos 3 necessários; sobra 1 bit de incerteza (entre d5 e d7).

### 2. Prever e Explorar
- Testar a pista "relevancia" em vez de "busca".

**Previsão:** "relevancia" também aparece em pelo menos 2 documentos pelo que vimos nas aulas anteriores (d1 e d7), então deve valer também 2 bits.

In [ ]:
contem("relevancia")
log2(8 / length(contem("relevancia")))

[1] "d1" "d7"
[1] 2


**Resultado:** confirmado: `relevancia` aparece em `d1` e `d7`, e vale exatamente 2 bits, igual a `busca`. Mas repare que os *documentos* que sobram são diferentes (d5/d7 vs d1/d7): dois termos podem valer o mesmo número de bits sem apontar para o mesmo subconjunto de candidatos.

## Bloco 3: A fórmula da informação em bits

In [ ]:
I <- function(p) -log2(p)
I(c(1, 1/2, 1/4, 1/8))  # certo, moeda, 2 moedas, 3 moedas

[1] 0 1 2 3


### 1. O que cada linha faz:
- `I <- function(p) -log2(p)`: implementa diretamente a fórmula da autoinformação de Shannon, `I(p) = -log2(p)`, que mede em bits a "surpresa" de um evento com probabilidade `p`.
- `I(c(1, 1/2, 1/4, 1/8))`: aplica a função a um vetor de probabilidades de uma vez (R vetoriza automaticamente). Resultado: evento certo (`p=1`) vale 0 bits; uma moeda justa (`p=1/2`) vale 1 bit; duas moedas independentes (`p=1/4`) valem 2 bits; três moedas (`p=1/8`) valem 3 bits, dobrando a "dificuldade" a cada bit extra.

### 2. Prever e Explorar
- Calcular a informação de um evento com `p = 1/3` (por exemplo, acertar de primeira um dado de 3 lados) e de um evento bem raro, `p = 0.01`.

**Previsão:** `p=1/3` deve ficar entre 1 e 2 bits (menos incerto que uma moeda dupla, mais que uma simples). `p=0.01` deve ser um número bem maior, porque é um evento raro.

In [ ]:
round(I(c(1/3, 0.01)), 2)

[1] 1.58 6.64


**Resultado:** `p=1/3` deu 1.58 bits (entre 1 e 2, como previsto) e `p=0.01` deu 6.64 bits, bem mais informativo, porque é um evento raro. Isso é exatamente a lógica que sustenta o IDF: termo raro (baixa probabilidade de aparecer) = mais bits = mais peso na busca.

## Bloco 4: A ponte entre o IDF e a autoinformação

In [ ]:
df <- c(modelo = 2, de = 5, recuperacao = 2)
round(log2(8 / df), 3)  # IDF em BITS

     modelo          de recuperacao 
      2.000       0.678       2.000 


### 1. O que cada linha faz:
- `df <- c(modelo = 2, de = 5, recuperacao = 2)`: fixa manualmente a *document frequency* (em quantos dos 8 documentos cada termo aparece) para três termos já conhecidos da Aula 01.
- `log2(8 / df)`: aplica a mesma fórmula `I(t) = log2(N/df_t)`; este é exatamente o IDF, só que em base 2 (bits) em vez do `log()` natural (nats) usado na Aula 01.
- O resultado mostra que "modelo" e "recuperacao" (2 docs cada) valem 2 bits cada, termos "raros" que cortam bastante o espaço de busca, enquanto "de" (5 docs, quase metade do corpus) vale só 0.678 bits: quase não estreita nada.

### 2. Prever e Explorar
- Adicionar um termo hipotético que aparece em 1 só documento (`df=1`, bem raro) e outro que aparece nos 8 (`df=8`, uma "stopword" perfeita).

**Previsão:** o termo com `df=1` deve valer o máximo possível, `log2(8/1)=3` bits (o total necessário para achar 1 entre 8). O termo com `df=8` deve valer 0 bits: presente em todos, não discrimina nada.

In [ ]:
df2 <- c(modelo = 2, de = 5, recuperacao = 2, bm25 = 1, todos = 8)
round(log2(8 / df2), 3)

     modelo          de recuperacao        bm25       todos 
      2.000       0.678       2.000       3.000       0.000 


**Resultado:** exatamente como previsto. `bm25` (que só aparece em `d3`) vale os 3 bits cheios: sozinho já identificaria o documento certo. E um termo hipotético em todos os 8 documentos vale 0: é a definição matemática de por que **stopwords têm IDF zero** e não ajudam em nada a distinguir documentos.

## Bloco 5: Quando a aditividade fecha certinho

In [ ]:
intersect(contem("busca"), contem("documentos"))
log2(8/2) + log2(8/4)  # 2 bits + 1 bit

[1] "d5"
[1] 3


### 1. O que cada linha faz:
- `intersect(contem("busca"), contem("documentos"))`: cruza os documentos que contêm "busca" (`d5`, `d7`) com os que contêm "documentos" (`d1`,`d2`,`d5`,`d6`); sobra só `d5`.
- `log2(8/2) + log2(8/4)`: soma a informação individual de cada termo (2 bits + 1 bit = 3 bits), e o total bate exatamente com os 3 bits necessários para achar 1 entre 8 documentos.
- Isso só acontece porque "busca" e "documentos" são estatisticamente **independentes** neste corpus (não co-ocorrem por afinidade temática, é "coincidência" estrutural).

### 2. Prever e Explorar
- Testar outro par independente: "documentos" + "recuperacao".

**Previsão:** se o par também fechar exatamente em 3 bits, é porque, como no caso anterior, os dois termos isolam o mesmo (único) documento sem se sobrepor em significado.

In [ ]:
intersect(contem("documentos"), contem("recuperacao"))
log2(8/4) + log2(8/2)  # 1 bit + 2 bits

[1] "d1"
[1] 3


**Resultado:** fechou de novo: `d1` é o único documento com os dois termos, e a soma deu exatamente 3 bits. Achei dois pares diferentes ('busca'+'documentos' e 'documentos'+'recuperacao') que fecham certinho porque, por acaso da forma como escrevi o corpus de brinquedo, esses termos não se correlacionam entre si além do necessário.

## Bloco 6: Quando a conta não fecha

In [ ]:
intersect(contem("recuperacao"), contem("relevancia"))
log2(8/2) + log2(8/2)  # 2 bits + 2 bits

[1] "d1"
[1] 4


### 1. O que cada linha faz:
- Mesma lógica do bloco anterior, mas agora com "recuperacao" e "relevancia": a interseção também é um único documento (`d1`), mas a soma das informações individuais dá **4 bits**, um bit a mais do que os 3 necessários.
- Isso acontece porque "recuperacao" e "relevancia" **não são independentes**: são do mesmo campo semântico (RI fala sobre relevância o tempo todo) e tendem a co-ocorrer mais do que o acaso preveria. A segunda pista já estava parcialmente "adiantada" pela primeira, então ela deveria valer menos do que seus 2 bits nominais.

### 2. Prever e Explorar
- Verificar se "avaliacao" + "relevancia" (também do mesmo campo semântico de RI) também "paga a mais".

**Previsão:** como ambos aparecem no vocabulário de avaliação de resultados de busca, espero que também superestimem: a soma deve ficar acima dos bits reais necessários para achar o documento comum.

In [ ]:
contem("avaliacao"); contem("relevancia")
intersect(contem("avaliacao"), contem("relevancia"))
log2(8/length(contem("avaliacao"))) + log2(8/length(contem("relevancia")))

[1] "d7"
[1] "d1" "d7"
[1] "d7"
[1] 4


**Resultado:** confirmado: a interseção também é um único documento (`d7`), mas a soma deu 4 bits em vez dos 3 necessários. Dois pares diferentes do mesmo campo semântico (relevância/avaliação vs. recuperação/relevância) mostram o mesmo padrão de superestimativa: é a mesma correlação temática se manifestando de novo, e não coincidência do par anterior.

## Bloco 7: Exemplo completo de consulta em bits

In [ ]:
idf_bits <- c(modelo = 2.00, de = 0.678, recuperacao = 2.00)
tokens <- lapply(docs, function(x) unlist(strsplit(tolower(x), " ")))
consulta <- c("modelo", "de", "recuperacao")

bits_doc <- sapply(tokens, function(tk)
  sum(sapply(consulta, function(t) sum(tk == t) * idf_bits[[t]]))
)
round(sort(bits_doc, decreasing = TRUE), 2)

   d3    d1    d2    d4    d6    d8    d5    d7 
 3.36  2.68  2.68  2.00  0.68  0.68  0.00  0.00 


### 1. O que cada linha faz:
- `idf_bits`: os IDFs em bits já calculados no Bloco 4 para "modelo", "de" e "recuperacao".
- `tokens <- lapply(...)`: tokeniza todo o corpus de uma vez, como na Aula 01.
- `bits_doc <- sapply(tokens, function(tk) sum(...))`: para cada documento, soma `tf(t,d) × IDF(t)` de cada termo da consulta, ou seja, calcula o **TF-IDF em bits** do documento inteiro para a consulta "modelo de recuperacao".
- `sort(..., decreasing = TRUE)`: ordena os documentos do que acumulou mais bits de evidência para o que acumulou menos; é literalmente o ranqueamento da busca.
- Repare em `d3`: ele lidera com 3.36 bits porque "de" aparece **duas vezes** nesse documento (0.68×2=1.36, mais os 2 bits de "modelo").

### 2. Prever e Explorar
- Trocar a consulta para "busca relevancia" (dois termos do campo de avaliação vistos no Bloco 6).

**Previsão:** `d7` deve liderar disparado, porque contém os dois termos ao mesmo tempo (vimos no Bloco 6 que a interseção de "avaliacao"/"relevancia" e agora "recuperacao"/"relevancia" sempre passa perto de `d7`/`d1`).

In [ ]:
idf_full <- function(t) log2(8 / length(contem(t)))
consulta2 <- c("busca", "relevancia")
idf_c2 <- sapply(consulta2, idf_full)

bits_doc2 <- sapply(tokens, function(tk)
  sum(sapply(consulta2, function(t) sum(tk == t) * idf_c2[[t]]))
)
round(sort(bits_doc2, decreasing = TRUE), 2)

 d7  d1  d5  d2  d3  d4  d6  d8 
  4   2   2   0   0   0   0   0 


**Resultado:** como previsto, `d7` lidera: só ele tem os dois termos ao mesmo tempo (4 bits, a soma completa de `log2(8/2)+log2(8/2)`), enquanto `d1` e `d5` empatam com 2 bits cada por terem só um dos dois termos ('relevancia' e 'busca', respectivamente).

## Bloco 8: O caso extremo das rotas

In [ ]:
rotas <- c(
  r1 = "santos cubatao guaruja bertioga",
  r2 = "santos guaruja cubatao bertioga",
  r3 = "santos bertioga guaruja cubatao",
  r4 = "cubatao santos bertioga guaruja",
  r5 = "guaruja bertioga santos cubatao",
  r6 = "bertioga cubatao santos guaruja",
  r7 = "guaruja santos cubatao bertioga",
  r8 = "bertioga guaruja cubatao santos"
)
tokens_r <- lapply(rotas, function(x) unlist(strsplit(x, " ")))
vocab_r <- sort(unique(unlist(tokens_r)))
tdm_r <- sapply(tokens_r, function(t) as.integer(table(factor(t, levels = vocab_r))))
rownames(tdm_r) <- vocab_r
tdm_r

         r1 r2 r3 r4 r5 r6 r7 r8
bertioga  1  1  1  1  1  1  1  1
cubatao   1  1  1  1  1  1  1  1
guaruja   1  1  1  1  1  1  1  1
santos    1  1  1  1  1  1  1  1


### 1. O que cada linha faz:
- `rotas`: 8 "documentos" artificiais, as mesmas 4 cidades da Baixada Santista, só que em ordens diferentes (permutações), simulando 8 rotas de entrega distintas.
- O resto reconstrói exatamente a TDM da Aula 01: tokeniza, monta vocabulário, conta ocorrências por documento.
- O resultado mostra que **toda célula da matriz vale 1**: cada cidade aparece exatamente uma vez em cada rota. Não há absolutamente nenhuma variação de contagem entre as rotas.

### 2. Prever e Explorar
- E se uma das rotas repetisse "santos" duas vezes (por exemplo, ida e volta pela mesma cidade) e não passasse por "guaruja"?

**Previsão:** essa rota deveria quebrar a uniformidade da matriz: as linhas "santos" e "guaruja" deixariam de ter `df=8`, então pelo menos esses dois termos passariam a ter algum IDF diferente de zero.

In [ ]:
rotas9 <- c(rotas, r9 = "santos santos cubatao bertioga")
tokens_r9 <- lapply(rotas9, function(x) unlist(strsplit(x, " ")))
vocab_r9 <- sort(unique(unlist(tokens_r9)))
tdm_r9 <- sapply(tokens_r9, function(t) as.integer(table(factor(t, levels = vocab_r9))))
rownames(tdm_r9) <- vocab_r9
tdm_r9
df_r9 <- rowSums(tdm_r9 > 0)
round(log(9 / df_r9), 3)

         r1 r2 r3 r4 r5 r6 r7 r8 r9
bertioga  1  1  1  1  1  1  1  1  1
cubatao   1  1  1  1  1  1  1  1  1
guaruja   1  1  1  1  1  1  1  1  0
santos    1  1  1  1  1  1  1  1  2
bertioga  cubatao  guaruja   santos 
   0.000    0.000    0.118    0.000 


**Resultado:** exatamente como previsto: `guaruja` deixou de aparecer em todos os documentos (`df=8` de 9), então ganhou um IDF pequeno mas não nulo (0.118 bits). `santos`, apesar de aparecer 2 vezes na rota 9, continua em `df=9` (df conta *documentos*, não ocorrências), então seu IDF continua zero. Isso reforça a diferença entre TF (quantas vezes) e DF (em quantos documentos).

## Bloco 9: Zero bits de informação deixam o motor cego

In [ ]:
df_r <- rowSums(tdm_r > 0)   # cada termo aparece em QUANTOS documentos?
idf_r <- log(8 / df_r)
df_r
idf_r

cosseno <- function(a, b) sum(a * b) / (sqrt(sum(a^2)) * sqrt(sum(b^2)))
cosseno(tdm_r[, "r1"], tdm_r[, "r8"])

bertioga  cubatao  guaruja   santos 
       8        8        8        8 
bertioga  cubatao  guaruja   santos 
       0        0        0        0 
[1] 1


### 1. O que cada linha faz:
- `df_r <- rowSums(tdm_r > 0)`: como toda cidade aparece em todas as 8 rotas, `df` é 8 para as 4 (o máximo possível).
- `idf_r <- log(8 / df_r)`: `log(8/8) = log(1) = 0` para todo termo; a matriz TF-IDF inteira seria zero.
- `cosseno(...)`: mede o ângulo entre os vetores de `r1` e `r8` (usando as contagens brutas, já que o TF-IDF é todo zero). O resultado é **1**: cosseno máximo, os vetores são idênticos.
- Isso mostra concretamente o que os slides descrevem: com 0 bits de informação disponível no vocabulário e 3 bits necessários para distinguir 1 rota entre 8, a busca é **matematicamente impossível**: o modelo de saco de palavras declara as 8 rotas iguais, mesmo elas sendo rotas diferentes na prática (a ordem é o que muda, e o saco de palavras descarta ordem).

### 2. Prever e Explorar
- Repetir o cálculo do cosseno usando o corpus de rotas com 9 documentos (Bloco 8), onde já quebramos a uniformidade.

**Previsão:** como `guaruja` já não tem mais `df=8`, o cosseno entre `r1` e `r9` (que não tem "guaruja") deve ser menor que 1; as duas rotas devem finalmente ficar distinguíveis.

In [ ]:
cosseno(tdm_r9[, "r1"], tdm_r9[, "r9"])
cosseno(tdm_r9[, "r1"], tdm_r9[, "r2"])  # duas rotas completas, para comparar

[1] 0.9682458
[1] 1


**Resultado:** confirmado: `r1` vs `r9` (que tem uma cidade a menos e outra repetida) deu cosseno ≈0.968, menor que 1. Já `r1` vs `r2` (duas rotas completas com as 4 cidades) continua em cosseno 1, idênticas para o modelo. A lição do slide se confirma na prática: **informação exige contraste**; sem variação de vocabulário entre documentos, não existe distância nenhuma para medir, não importa a ordem real das palavras.

---
# Parte 2: Investigar

Responda usando o corpus de 8 documentos (`docs`) ou o de rotas (`rotas`), conforme o caso.


### 1. Se o corpus tivesse 1024 documentos, quantos bits custaria achar um? E se dobrasse para 2048?

In [ ]:
log2(1024)
log2(2048)

[1] 10
[1] 11


**Resposta:** 10 bits para 1024 documentos e 11 bits para 2048. Dobrar o corpus sempre custa exatamente **+1 bit**, não o dobro; é a própria definição de logaritmo na base 2: cada bit a mais dobra o espaço de busca que ele consegue distinguir. Passar de 8 (3 bits) para 1024 (10 bits) é *7 bits a mais* para um corpus **128 vezes maior**, não 128 vezes mais bits.

### 2. Encontre um par de termos cuja soma de bits superestime a informação real. Explique a correlação.

In [ ]:
contem("avaliacao"); contem("relevancia")
intersect(contem("avaliacao"), contem("relevancia"))
log2(8/length(contem("avaliacao"))) + log2(8/length(contem("relevancia")))

[1] "d7"
[1] "d1" "d7"
[1] "d7"
[1] 4


**Resposta:** "avaliacao" (aparece só em `d7`) e "relevancia" (aparece em `d1` e `d7`) se cruzam em `d7`, mas a soma das informações individuais dá 4 bits, um a mais que os 3 necessários para achar 1 em 8. A explicação é a mesma do Bloco 6: os dois termos pertencem ao **mesmo campo semântico** (avaliação de relevância em RI, o assunto do próprio `d7`: "a avaliacao mede a relevancia dos resultados da busca"). Termos do mesmo assunto tendem a co-ocorrer mais do que o acaso preveria; então, ao ver o segundo termo, o leitor (ou o modelo) já estava "menos surpreso" do que os bits nominais sugerem. A fórmula de aditividade (`I(p1·p2) = I(p1)+I(p2)`) só vale para eventos **independentes**, e esses dois termos não são.

### 3. Troque `log2` por `log` no cálculo do IDF. O ranking muda? Por quê?

In [ ]:
df3 <- c(modelo = 2, de = 5, recuperacao = 2)
idf_log2 <- log2(8 / df3)
idf_ln   <- log(8 / df3)
round(idf_log2, 3)
round(idf_ln, 3)
# a ORDEM relativa entre os termos:
order(idf_log2, decreasing = TRUE)
order(idf_ln,   decreasing = TRUE)

     modelo          de recuperacao 
      2.000       0.678       2.000 
     modelo          de recuperacao 
      1.386       0.470       1.386 
[1] 1 3 2
[1] 1 3 2


**Resposta:** os valores absolutos mudam (bits vs. nats: `log()` em R é logaritmo natural, base *e*), mas a **ordem relativa entre os termos não muda**: "modelo" e "recuperacao" continuam empatados em primeiro lugar, "de" continua por último, em ambas as bases. Isso acontece porque trocar a base de um logaritmo é multiplicar todo o resultado por uma constante (`log2(x) = log(x)/log(2)`), e multiplicar todos os pesos por uma mesma constante positiva **preserva o ranking**. Por isso o TF-IDF da Aula 01 (com `log()`) e o desta aula (com `log2()`) concordam sobre qual documento é mais relevante; só discordam na unidade de medida.

### 4. Um termo aparece em todos os documentos. Quantos bits ele vale? O que isso diz sobre stopwords?

In [ ]:
log2(8 / 8)

[1] 0


**Resposta:** 0 bits: é o mínimo absoluto de informação, o mesmo valor de um "evento certo" (`I(1)=0` no Bloco 3). Um termo presente em 100% dos documentos não ajuda a distinguir nenhum deles: saber que o documento "contém a palavra X" não estreita em nada o conjunto de candidatos, se X está em todos. Essa é exatamente a justificativa matemática, e não apenas uma regra prática, para remover **stopwords** ("de", "a", "o", "que"...) antes de indexar: elas tendem a ter `df` altíssimo (perto de N), logo IDF próximo de zero, logo contribuem quase nada ao TF-IDF e só ocupam espaço/processamento no índice.